# FIFA World Cup Analysis

Analysis of World Cup data from1930 to2026.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns',50)

In [ ]:
# Load data
hist=pd.read_csv('WC_1930-2014.csv').dropna(subset=['Year'])
hist['Year']=hist['Year'].astype(int)
hist['total_goals']=hist['Home Team Goals']+hist['Away Team Goals']

matches_2018=pd.read_csv('World_cup_2018_matches.csv')
goals_2018=pd.read_csv('World_cup_2018_goals.csv')
countries_2018=pd.read_csv('World_cup_2018_country.csv')

matches_2022=pd.read_csv('WC_2022.csv')

teams_2026=pd.read_csv('teams.csv')
matches_2026=pd.read_csv('matches.csv')
venues_2026=pd.read_csv('venues.csv')

print(f'Historical: {len(hist)} matches')
print(f'2018: {len(matches_2018)} matches, {len(goals_2018)} goals')
print(f'2022: {len(matches_2022)} matches')
print(f'2026: {len(matches_2026)} matches, {len(teams_2026)} teams')

In [ ]:
# Historical trends
yearly=hist.groupby('Year').agg(
    matches=('MatchID','count'),
    goals=('total_goals','sum'),
    gpg=('total_goals','mean')
).reset_index()

fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].bar(yearly['Year'],yearly['matches'],color='steelblue')
axes[0].set_title('Matches per Tournament')
axes[1].plot(yearly['Year'],yearly['gpg'],'o-',color='green')
axes[1].set_title('Goals per Game')
plt.tight_layout()
plt.show()

In [ ]:
#2018 analysis
matches_2018['total_goals']=matches_2018['Home_goals']+matches_2018['Away_goals']

fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].hist(matches_2018['total_goals'],bins=range(0,10),color='green',alpha=0.7)
axes[0].set_title('Goals Distribution2018')

scorers=goals_2018['Player_scored'].value_counts().head(10)
axes[1].barh(scorers.index[::-1],scorers.values[::-1],color='gold')
axes[1].set_title('Top Scorers2018')
plt.tight_layout()
plt.show()

In [ ]:
#2022 team performance
def agg2022(df):
    teams={}
    for _,r in df.iterrows():
        for tc in ['team1','team2']:
            t=r[tc]
            if t not in teams:teams[t]={'gf':0,'ga':0,'poss':[]}
            p='1' if tc=='team1' else '2'
            teams[t]['gf']+=r[f'number of goals {tc}']
            teams[t]['ga']+=r[f'number of goals team{"2" if p=="1" else "1"}']
            teams[t]['poss'].append(r[f'possession {tc}'])
    return pd.DataFrame([{'Team':t,'GF':s['gf'],'GA':s['ga'],'GD':s['gf']-s['ga'],'Poss':np.mean(s['poss'])} for t,s in teams.items()]).sort_values('GD',ascending=False)

t22=agg2022(matches_2022)
print('Top5 teams by goal difference:')
display(t22.head())

In [ ]:
#2026 overview
print(f'2026 World Cup: {len(teams_2026)} teams, {len(matches_2026)} matches')
print(f'\nGroups: {teams_2026["group_letter"].nunique()}')
print(f'Venues: {len(venues_2026)}')

conf=teams_2026['confederation'].value_counts()
print('\nConfederation representation:')
print(conf)